In [12]:
spark.sql("SHOW CATALOGS").show(truncate=False)
spark.sql("USE CATALOG cattesteporto")
spark.sql("CREATE DATABASE IF NOT EXISTS soe")
spark.sql("USE soe")


+-------------+--------+-----------+----------------------------------------------------------------------------+-------------+-----------------+-------------------+
|catalog_name |type    |source_type|created_by                                                                  |created_at   |last_refresh_time|last_refresh_status|
+-------------+--------+-----------+----------------------------------------------------------------------------+-------------+-----------------+-------------------+
|ceal_porto   |INTERNAL|NA         |ocid1.user.oc1..aaaaaaaa3kybcpxpzkr73dg4lbhzdlqkl5eburhi7ls3mh34zrpgfuuccqfq|1770834590575|NA               |NA                 |
|adb01        |EXTERNAL|ORACLE_ALH |ocid1.user.oc1..aaaaaaaa3kybcpxpzkr73dg4lbhzdlqkl5eburhi7ls3mh34zrpgfuuccqfq|1770003413525|NA               |FAILED             |
|catalogo01   |INTERNAL|NA         |ocid1.user.oc1..aaaaaaaa3kybcpxpzkr73dg4lbhzdlqkl5eburhi7ls3mh34zrpgfuuccqfq|1769999317469|NA               |NA                 |
|cat

DataFrame[]

In [13]:
from datetime import datetime
import re
from pyspark.sql.functions import col
from pyspark.sql import SparkSession

#spark = SparkSession.builder.appName("SOE.CUSTOMERS").getOrCreate()

# Variables from original code (assume defined elsewhere)
url = "jdbc:oracle:thin:@(description=(address=(host=204.216.172.140)(protocol=tcp)(port=1521))(connect_data=(SERVICE_NAME=TESTE)))?oracle.jdbc.ReadTimeout=1800000" 
adwuser = "soe"
password = "WORKSHOPsec2019##" 
table = "SOE.CUSTOMERS"
 
# Split owner and table_name
owner, table_name = table.split('.')
 
base_options = {
    "url": url,
    "user": adwuser,
    "password": password,
    "driver": "oracle.jdbc.driver.OracleDriver"
}
 
# Step 1: Find best partition column (numeric, integer, highest num_distinct)
query = f"""
SELECT c.COLUMN_NAME, c.NUM_DISTINCT,
       (SELECT COUNT(*) FROM ALL_IND_COLUMNS i WHERE i.TABLE_OWNER = c.OWNER AND i.TABLE_NAME = c.TABLE_NAME AND i.COLUMN_NAME = c.COLUMN_NAME) AS INDEX_COUNT
FROM ALL_TAB_COLUMNS c
WHERE c.OWNER = '{owner}'
AND c.TABLE_NAME = '{table_name}'
AND c.DATA_TYPE = 'NUMBER'
AND (c.DATA_SCALE = 0 OR c.DATA_SCALE IS NULL)
ORDER BY NUM_DISTINCT DESC, INDEX_COUNT DESC
"""
 
partition_df = spark.read.format("jdbc").options(**base_options).option("dbtable", f"({query})").load()
 
if partition_df.count() == 0:
    # No suitable partition column, read without partitioning
    fetchsize = 100000
    df = spark.read \
        .format("jdbc") \
        .options(**base_options) \
        .option("dbtable", table) \
        .option("fetchsize", fetchsize) \
        .load()
else:
    # Select the column with highest cardinality
    row = partition_df.collect()[0]
    partitionColumn = row['COLUMN_NAME']
    num_distinct = row['NUM_DISTINCT']

    # --- AJUSTE NA ETAPA 2: Garantir que min/max sejam inteiros ---
    min_max_query = f"SELECT CAST(MIN({partitionColumn}) AS NUMBER(19)) AS MIN_VAL, CAST(MAX({partitionColumn}) AS NUMBER(19)) AS MAX_VAL FROM {table}"
    min_max_df = spark.read.format("jdbc").options(**base_options).option("dbtable", f"({min_max_query})").load()
    mm_row = min_max_df.collect()[0]
    
    # Convertendo explicitamente para int no Python para remover decimais
    lowerBound = int(mm_row['MIN_VAL'])
    upperBound = int(mm_row['MAX_VAL']) + 1 

    # --- AJUSTE NA ETAPA 3: Garantir que num_rows seja inteiro ---
    num_rows_query = f"SELECT NUM_ROWS FROM ALL_TABLES WHERE OWNER = '{owner}' AND TABLE_NAME = '{table_name}'"
    num_rows_df = spark.read.format("jdbc").options(**base_options).option("dbtable", f"({num_rows_query})").load()
    num_rows = int(num_rows_df.collect()[0]['NUM_ROWS'] or 1)

    # --- AJUSTE NA ETAPA 4: Cast explícito para string nos parâmetros JDBC ---
    numPartitions = max(1, min(int(num_rows / 1000000), int(num_distinct / 10)))
    if numPartitions < 2:
        numPartitions = 12

    rows_per_partition = num_rows // numPartitions
    fetchsize = max(10000, min(1000000, int(rows_per_partition / 10)))

    # Step 6: Load the data com as variáveis tratadas
    df = spark.read \
        .format("jdbc") \
        .options(**base_options) \
        .option("dbtable", table) \
        .option("fetchsize", str(fetchsize)) \
        .option("partitionColumn", partitionColumn) \
        .option("lowerBound", str(lowerBound)) \
        .option("upperBound", str(upperBound)) \
        .option("numPartitions", str(numPartitions)) \
        .load()

df.show(5)
df.write \
  .mode("overwrite") \
  .saveAsTable("soe.customers")


Command ID 446a5e6b-fe75-45f7-87c6-243706e92c90 failed with java.lang.RuntimeException: [Command 446a5e6b-fe75-45f7-87c6-243706e92c90 in Context 413b26fb-47ae-4384-a3d4-9d5ffffc15d0] failed with error: 
 ---------------------------------------------------------------------------Py4JJavaError                             Traceback (most recent call last)Cell In[9], line 36
     24 # Step 1: Find best partition column (numeric, integer, highest num_distinct)
     25 query = f"""
     26 SELECT c.COLUMN_NAME, c.NUM_DISTINCT,
     27        (SELECT COUNT(*) FROM ALL_IND_COLUMNS i WHERE i.TABLE_OWNER = c.OWNER AND i.TABLE_NAME = c.TABLE_NAME AND i.COLUMN_NAME = c.COLUMN_NAME) AS INDEX_COUNT
   (...)
     33 ORDER BY NUM_DISTINCT DESC, INDEX_COUNT DESC
     34 """
---> 36 partition_df = spark.read.format("jdbc").options(**base_options).option("dbtable", f"({query})").load()
     38 if partition_df.count() == 0:
     39     # No suitable partition column, read without partitioning
     40     

In [11]:
spark.sql("select count(*) from soe.customers").show(truncate=False)

+--------+
|count(1)|
+--------+
|20000000|
+--------+



In [ ]:
novo teste